# NB 05 — LSTM Next-Event Model: Architecture, Inference Parity and Decision Checks

In [1]:
import json
import os
import platform
import random
import sys
import threading
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from zentinel.config import ARTIFACTS_DIR, DATA_DIR, REPORTS_DIR  # shared path constants

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
try:
    torch.use_deterministic_algorithms(False)  # CPU inference deterministic anyway
    _ = torch.use_deterministic_algorithms
except AttributeError:  # pragma: no cover
    pass

try:
    import plotly.express as px
    import plotly.graph_objects as go
    import plotly.io as pio

    pio.renderers.default = "plotly_mimetype+notebook"
    PLOTLY_AVAILABLE = True
except Exception as exc:  # pragma: no cover - informational only
    PLOTLY_AVAILABLE = False
    PX_IMPORT_ERROR = str(exc)

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("ROOT:", ROOT)
print(
    "python:", platform.python_version(),
    "| numpy:", np.__version__,
    "| pandas:", pd.__version__,
    "| torch:", torch.__version__,
)
print("seed:", RANDOM_SEED)


ROOT: C:\Users\noupa\Desktop\datathon
python: 3.11.9 | numpy: 2.4.6 | pandas: 3.0.6 | torch: 2.14.1+cpu
seed: 42


## Purpose

Inspect and exercise the **production LSTM next-event predictor** — the same checkpoint the live API scores with:

- Architecture: embedding(616→32) → LSTM(32→64) → linear(64→616); checkpoint config is verified below.
- Semantics: the model predicts the *next event token*; a window's score is the mean next-event cross-entropy. Production scoring traverses **contiguous known-token runs** (UNK/PAD break a run; runs < 2 contribute no transitions).
- Threshold 3.4791 was selected on **validation only**; thresholds are never derived from test data.
- Training details (from the committed report): seed 42, 8 epochs, lr 1e-3, trained on normal training windows only.

Checks: architecture, real inference on real sequences, score parity against persisted production scores, and end-to-end decision parity.

## 1. Load checkpoint and verify architecture/config integrity

In [2]:
checkpoint = torch.load(MODEL_PATH := ROOT / "artifacts" / "lstm_next_event.pt", map_location="cpu", weights_only=True)
ckpt_config = checkpoint["config"]
print(json.dumps(ckpt_config, indent=2, default=str))
assert ckpt_config["model_type"] == "LSTMNextEvent"
assert int(ckpt_config["vocab_size"]) == 616

{
  "model_type": "LSTMNextEvent",
  "vocab_size": 616,
  "embedding_dim": 32,
  "hidden_dim": 64,
  "window_size": 100,
  "padding_token": 0,
  "unknown_token": 1,
  "seed": 42,
  "epochs": 8,
  "learning_rate": 0.001,
  "trained_on_normal_training_windows_only": true
}


In [3]:
from zentinel.models.inference import LSTMNextEvent

model = LSTMNextEvent(vocab_size=int(ckpt_config["vocab_size"]),
                      embedding_dim=int(ckpt_config["embedding_dim"]),
                      hidden_dim=int(ckpt_config["hidden_dim"]))
model.load_state_dict(checkpoint["model_state_dict"], strict=True)
model.eval()
param_count = sum(p.numel() for p in model.parameters())
state_shapes = {k: tuple(v.shape) for k, v in model.state_dict().items()}
print(f"params: {len(state_shapes)} tensors, total={param_count}")
print(state_shapes)
expected_first = ("embedding.weight", (616, 32))
assert state_shapes["embedding.weight"] == expected_first[1], "embedding shape changed"
assert param_count > 80_000, "parameter contract sanity"

params: 7 tensors, total=84840
{'embedding.weight': (616, 32), 'lstm.weight_ih_l0': (256, 32), 'lstm.weight_hh_l0': (256, 64), 'lstm.bias_ih_l0': (256,), 'lstm.bias_hh_l0': (256,), 'output.weight': (616, 64), 'output.bias': (616,)}


In [4]:
history_report = json.loads((REPORTS_DIR / "lstm_validation_report.json").read_text(encoding="utf-8"))
print("model_config from committed report:", json.dumps(history_report["model_config"], indent=2))
assert history_report["model_config"]["seed"] == RANDOM_SEED == 42
assert history_report["threshold_selection"]["selected_using_validation_only"] is True
assert history_report["threshold_selection"]["test_used_for_threshold_selection"] is False
print("training constraint:", history_report["threshold_selection"]["score_definition"])

model_config from committed report: {
  "model_type": "LSTMNextEvent",
  "vocab_size": 616,
  "embedding_dim": 32,
  "hidden_dim": 64,
  "window_size": 100,
  "padding_token": 0,
  "unknown_token": 1,
  "seed": 42,
  "epochs": 8,
  "learning_rate": 0.001,
  "trained_on_normal_training_windows_only": true
}
training constraint: Mean next-event cross-entropy per valid sequence position


## 2. Real inference on real sequences (validation split)

In [5]:
import torch.nn.functional as F

seq = np.load(DATA_DIR / "processed" / "bgl_event_sequences_v2.npz")
code_of = {0: "train", 1: "validation", 2: "test"}
split_of = np.array([code_of[c] for c in seq["split_codes"]])
val_idx = np.flatnonzero(split_of == "validation")[:1000]
val_seq = torch.from_numpy(seq["sequences"][val_idx].astype("int64"))
val_target = seq["targets"][val_idx]

with torch.inference_mode():
    inputs = val_seq[:, :-1]                       # next-event convention: logits at position i predict token i+1
    targets = val_seq[:, 1:]
    logits = model(inputs)                         # (B, L-1, V)
    losses = F.cross_entropy(logits.transpose(1, 2), targets, reduction="none")  # (B, L-1)
    # Production mask: per window length L, positions < L-1 are scored (padding targets excluded).
    lengths = torch.from_numpy(seq["lengths"][val_idx].astype("int64")).unsqueeze(1)
    positions = torch.arange(targets.shape[1]).unsqueeze(0)
    valid = positions < (lengths - 1)
    window_scores = (losses.masked_fill(~valid, 0).sum(dim=1) / valid.sum(dim=1).clamp(min=1)).numpy()

local = pd.DataFrame({"window_index": seq["window_ids"][val_idx], "local_score": window_scores, "target_anomaly": val_target})
prod = pd.read_parquet(DATA_DIR / "processed" / "lstm_validation_scores.parquet")
merged = local.merge(prod[prod["split"] == "validation"][["window_id", "anomaly_score", "predicted_anomaly"]],
                     left_on="window_index", right_on="window_id", how="inner")
merged["abs_diff"] = (merged["local_score"] - merged["anomaly_score"]).abs()
merged["decision_agreement"] = (
    (merged["local_score"] >= float(history_report["threshold_selection"]["threshold"])).astype(int)
    == merged["predicted_anomaly"].astype(int)
)
print("score parity: max abs diff =", float(merged["abs_diff"].max()))
print("threshold decision agreement:", float(100 * merged["decision_agreement"].mean()), "%")
assert float(merged["abs_diff"].max()) < 2e-3, "score parity against committed production scores"
assert float(merged["decision_agreement"].mean()) >= 0.99
merged.tail(5)

score parity: max abs diff = 1.9073486328125e-06
threshold decision agreement: 100.0 %


,window_index,local_score,target_anomaly,window_id,anomaly_score,predicted_anomaly,abs_diff,decision_agreement
995,34231,0.043257,0,34231,0.043257,0,2.756715e-07,True
996,34232,0.043257,0,34232,0.043257,0,2.756715e-07,True
997,34233,0.043257,0,34233,0.043257,0,2.756715e-07,True
998,34234,0.043257,0,34234,0.043257,0,2.756715e-07,True
999,34235,0.043257,0,34235,0.043257,0,2.756715e-07,True


## 3. Known-run scoring path (production semantics)

In [6]:
from zentinel.models.inference import InferenceEngine

ENGINE = InferenceEngine()
ENGINE.load()
engine = ENGINE

prod_test_knownrun = pd.read_parquet(DATA_DIR / "processed" / "lstm_known_run_test_scores.parquet")
canvas = prod_test_knownrun.head(200).copy()
print("matched test windows for known-run parity:", len(canvas))
known_run_checks = []
for row in canvas.itertuples():
    # npz window_ids are identity-mapped: global window id == row index in the sequence array
    tokens = seq["sequences"][int(row.window_index)].tolist()
    score, transitions = engine.score_known_runs(tokens)
    known_run_checks.append({
        "window_index": int(row.window_index),
        "production_transitions": int(row.scored_transitions),
        "recomputed_transitions": int(transitions),
        "production_score": float(row.anomaly_score) if pd.notna(row.anomaly_score) else None,
        "recomputed_score": float(score) if score is not None else None,
    })
kr = pd.DataFrame(known_run_checks)
matched_transitions = int((kr["production_transitions"] == kr["recomputed_transitions"]).sum())
score_diffs = (kr["production_score"] - kr["recomputed_score"]).abs().dropna()
print("transition-count parity:", matched_transitions, "/", len(kr))
print("known-run score parity: max abs diff =", float(score_diffs.max()) if len(score_diffs) else "n/a")
assert matched_transitions == len(kr), "known-run semantics must reproduce production transition counts"
assert float(score_diffs.max()) < 2e-3

matched test windows for known-run parity: 200


transition-count parity: 200 / 200
known-run score parity: max abs diff = 3.0825836505599113e-07


## 4. End-to-end three-way decision parity

In [7]:
decisions = pd.read_parquet(DATA_DIR / "processed" / "lstm_three_way_test_decisions.parquet")
sample = decisions[decisions["model_scorable"]].head(150)
decision_checks = []
for row in sample.itertuples():
    seq_row = seq["sequences"][int(row.window_index)]  # global window id == array row index
    events = [
        {"token": int(tok), "line_number": i, "timestamp": 0, "parse_status": "x",
         "event_status": "matched" if tok > 1 else "unknown_template", "matched_event_id": None, "template": "t"}
        for i, tok in enumerate(seq_row)
        if tok > 0  # drop padded trailing positions
    ]
    d = engine.decide_window(events)
    decision_checks.append({
        "window_index": int(row.window_index),
        "production_decision": row.decision,
        "recomputed_decision": d["decision"],
        "matches": d["decision"] == row.decision,
        "score_matches": bool(
            d["lstm_anomaly_score"] is None and pd.isna(row.lstm_anomaly_score)
            or (d["lstm_anomaly_score"] is not None and
                abs(d["lstm_anomaly_score"] - float(row.lstm_anomaly_score)) < 2e-3)
        ),
    })
dc = pd.DataFrame(decision_checks)
agreement = float(dc["matches"].mean())
print(f"three-way decision agreement on {len(dc)} real test windows: {agreement:.3%}")
assert agreement >= 0.97, "decision parity drifted: investigate model/threshold loading"
dc.head(8)

three-way decision agreement on 150 real test windows: 100.000%


,window_index,production_decision,recomputed_decision,matches,score_matches
0,40358,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True
1,40359,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True
2,40360,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True
3,40361,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True
4,40362,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True
5,40363,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True
6,40364,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True
7,40365,NO_ANOMALY_DETECTED,NO_ANOMALY_DETECTED,True,True


### Zero-vocabulary symmetry check

`score_known_runs` treats tokens 0 (PAD) and 1 (UNKNOWN) as run breakers; the recomputed decisions in the previous cell also consume the same drift rule (unknown rate > 0.02). Decision parity therefore validates *model weights + threshold + drift rule together*.

## 5. Inference latency on this host (informational, not a LinuxONE claim)

In [8]:
lat_tokens = seq["sequences"][np.flatnonzero(split_of == "test")[:200]]
t0 = time.perf_counter()
with torch.inference_mode():
    for arr in lat_tokens:
        engine.score_known_runs(arr.tolist())
elapsed = time.perf_counter() - t0
latency_stats = {"windows": int(len(lat_tokens)), "seconds": round(elapsed, 3),
                 "ms_per_window": round(1000 * elapsed / len(lat_tokens), 3),
                 "host": platform.platform()}
print(json.dumps(latency_stats, indent=2))

{
  "windows": 200,
  "seconds": 0.094,
  "ms_per_window": 0.469,
  "host": "Windows-10-10.0.26300-SP0"
}


## 6. Persist the audit report

In [9]:
nb05_report = {
    "notebook": "05_lstm_next_event_model",
    "generated_at_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "checkpoint": "artifacts/lstm_next_event.pt",
    "checkpoint_sha256": __import__("hashlib").sha256(MODEL_PATH.read_bytes()).hexdigest(),
    "checkpoint_config": ckpt_config,
    "committed_report_config": history_report["model_config"],
    "window_score_parity_max_abs_diff": float(merged["abs_diff"].max()),
    "known_run_transition_parity": f"{matched_transitions}/{len(kr)}",
    "known_run_score_parity_max_abs_diff": (float(score_diffs.max()) if len(score_diffs) else None),
    "three_way_decision_agreement_on_test_sample": agreement,
    "inference_latency_this_host": latency_stats,
    "constraints": [
        "threshold selected on validation only; exploratory test used for diagnostics",
        "production scoring uses known-run transitions (UNK/PAD break runs)",
        "no refit of production artifacts anywhere in this lab",
    ],
}
out = REPORTS_DIR / "nb05_lstm_checks.json"
out.write_text(json.dumps(nb05_report, indent=2, default=str), encoding="utf-8")
print("wrote", out)

RUN_OK = True
print('NOTEBOOK_COMPLETE')  # sentinel: proves full execution reached the end

wrote C:\Users\noupa\Desktop\datathon\reports\nb05_lstm_checks.json
NOTEBOOK_COMPLETE
